In [1]:

%pip install -q optuna xgboost tensorflow scikit-learn plotly pandas statsmodels joblib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:

import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import seasonal_decompose

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

optuna.logging.set_verbosity(optuna.logging.WARNING)
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)
print("XGBoost:", xgb.__version__)


TensorFlow: 2.20.0
Optuna: 4.9.0
XGBoost: 3.2.0


In [ ]:
PROJECT_DIR = os.path.join(os.getcwd(), "gold_price_project")
PATHS = {
    "base": PROJECT_DIR,
    "models_dir": os.path.join(PROJECT_DIR, "models"),
    "data_dir": os.path.join(PROJECT_DIR, "data"),
    "optuna_dir": os.path.join(PROJECT_DIR, "optuna"),
    "raw_csv": os.path.join(PROJECT_DIR, "data", "gold_prices.csv"),
    "features_csv": os.path.join(PROJECT_DIR, "data", "features.csv"),
    "scaler_path": os.path.join(PROJECT_DIR, "models", "minmax_scaler.pkl"),
    "lstm_model_path": os.path.join(PROJECT_DIR, "models", "lstm_model.keras"),
    "gru_model_path": os.path.join(PROJECT_DIR, "models", "gru_model.keras"),
    "xgb_model_path": os.path.join(PROJECT_DIR, "models", "xgb_model.json"),
    "lstm_history_path": os.path.join(PROJECT_DIR, "models", "lstm_history.json"),
    "gru_history_path": os.path.join(PROJECT_DIR, "models", "gru_history.json"),
    "lstm_best_params_path": os.path.join(PROJECT_DIR, "models", "lstm_best_params.json"),
    "gru_best_params_path": os.path.join(PROJECT_DIR, "models", "gru_best_params.json"),
    "xgb_best_params_path": os.path.join(PROJECT_DIR, "models", "xgb_best_params.json"),
    "metrics_path": os.path.join(PROJECT_DIR, "models", "metrics.json"),
    "optuna_lstm_db": os.path.join(PROJECT_DIR, "optuna", "lstm_study.db"),
    "optuna_gru_db": os.path.join(PROJECT_DIR, "optuna", "gru_study.db"),
    "test_predictions_path": os.path.join(PROJECT_DIR, "models", "test_predictions.json"),
    "feature_columns_path": os.path.join(PROJECT_DIR, "models", "feature_columns.json"),
    "train_meta_path": os.path.join(PROJECT_DIR, "models", "train_meta.json"),
}
print("مسیر پروژه:", PROJECT_DIR)


In [ ]:
#پاکسازی داده‌ها و مدیریت مقادیر گمشده

RAW_NUMERIC_COLS = ["Close/Last", "Volume", "Open", "High", "Low"]
TARGET_COL = "Close/Last"


def load_raw_csv(path):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    if df["Date"].isna().any():
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)
    return df


def missing_value_report(df, cols=None):
    cols = cols or df.columns.tolist()
    n = len(df)
    rows = []
    for c in cols:
        na = int(df[c].isna().sum())
        rows.append({
            "ستون": c,
            "تعداد گمشده": na,
            "درصد گمشده": round(100 * na / n, 3) if n else 0.0,
        })
    return pd.DataFrame(rows)


def clean_missing_values(df):
    df = df.copy()
    if TARGET_COL in df.columns:
        df[TARGET_COL] = df[TARGET_COL].interpolate(method="linear", limit_direction="both")

    other_cols = [c for c in RAW_NUMERIC_COLS if c != TARGET_COL and c in df.columns]
    for c in other_cols:
        df[c] = df[c].ffill().bfill()

    return df


#Feature Engineering

LAG_DAYS = [1, 7, 30]
ROLLING_WINDOW = 7


def add_technical_features(df):
    df = df.copy()

    for lag in LAG_DAYS:
        df[f"lag_{lag}"] = df[TARGET_COL].shift(lag)

    df["rolling_mean_7"] = df[TARGET_COL].rolling(window=ROLLING_WINDOW).mean()
    df["rolling_std_7"] = df[TARGET_COL].rolling(window=ROLLING_WINDOW).std()

    df["daily_return"] = df[TARGET_COL].pct_change() * 100.0

    df["hl_ratio"] = df["High"] / df["Low"]

    df["day_of_week"] = df["Date"].dt.dayofweek
    df["month"] = df["Date"].dt.month
    df["day_of_year"] = df["Date"].dt.dayofyear

    return df


def add_rolling_stats_for_eda(df, target_col=TARGET_COL):
    df = df.copy()
    df["MA_7"] = df[target_col].rolling(window=7).mean()
    df["MA_30"] = df[target_col].rolling(window=30).mean()
    return df


def build_feature_frame(df_clean):
    feat = add_technical_features(df_clean)
    feat = feat.dropna().reset_index(drop=True)
    return feat


def get_model_feature_columns():
    return [
        "Open", "High", "Low", "Volume",
        "lag_1", "lag_7", "lag_30",
        "rolling_mean_7", "rolling_std_7",
        "daily_return", "hl_ratio",
        "day_of_week", "month", "day_of_year",
    ]


def train_test_split_timeseries(df, train_ratio=0.8):
    n = len(df)
    split_idx = int(n * train_ratio)
    train_df = df.iloc[:split_idx].reset_index(drop=True)
    test_df = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df


# ساخت Sequence برای LSTM/GRU

def create_sequences(data, seq_length=30, target_idx=0):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i - seq_length:i, :])
        y.append(data[i, target_idx])
    return np.array(X), np.array(y)


#Evaluation Metrics

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    nonzero_mask = y_true != 0
    mape = float(np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100.0)

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}


#تحلیل احساسات بازار

def compute_simulated_sentiment(df, lookback=30):
    d = df.tail(lookback).copy()
    if len(d) < 5:
        return {"score": 0.0, "label": "خنثی", "label_en": "Neutral"}

    last_return = d["daily_return"].iloc[-1] if "daily_return" in d.columns else 0.0
    last_return_norm = np.clip(last_return / 5.0, -3, 3)  

    vol_mean = d["Volume"].mean()
    vol_std = d["Volume"].std()
    volume_z = (d["Volume"].iloc[-1] - vol_mean) / vol_std if vol_std and not np.isnan(vol_std) and vol_std != 0 else 0.0
    volume_z = np.clip(volume_z, -3, 3)

    ma = d[TARGET_COL].mean()
    last_price = d[TARGET_COL].iloc[-1]
    price_position = (last_price - ma) / ma if ma != 0 else 0.0
    price_position_norm = np.clip(price_position * 20, -3, 3)  

    raw = last_return_norm * 0.4 + volume_z * 0.3 + price_position_norm * 0.3
    score = float(np.tanh(raw))

    if score > 0.15:
        label, label_en = "مثبت (صعودی)", "Positive (Bullish)"
    elif score < -0.15:
        label, label_en = "منفی (نزولی)", "Negative (Bearish)"
    else:
        label, label_en = "خنثی", "Neutral"

    return {
        "score": round(score, 4),
        "label": label,
        "label_en": label_en,
        "components": {
            "last_return_norm": round(float(last_return_norm), 4),
            "volume_z_score": round(float(volume_z), 4),
            "price_position_vs_MA": round(float(price_position_norm), 4),
        },
    }


# Ensemble + Explain

def combine_prediction(model_prediction, sentiment_score, average_price_change):
    return model_prediction + (0.25 * sentiment_score * average_price_change)


def generate_explanation(
    predicted_price,
    last_price,
    ma_30,
    ma_7,
    volume,
    avg_volume,
    sentiment,
    lang="fa",
):
    change = predicted_price - last_price
    change_pct = (change / last_price * 100.0) if last_price else 0.0
    direction_up = change > 0

    above_ma30 = last_price > ma_30
    high_volume = volume > avg_volume

    confidence = min(95, max(50, 60 + abs(change_pct) * 5 + (10 if abs(sentiment["score"]) > 0.15 else 0)))

    if lang == "fa":
        if above_ma30 and high_volume:
            technical_reason = "قرارگیری قیمت بالای میانگین متحرک ۳۰ روزه همراه با حجم بالای معاملات"
        elif above_ma30:
            technical_reason = "قرارگیری قیمت بالای میانگین متحرک ۳۰ روزه"
        elif high_volume:
            technical_reason = "حجم بالای معاملات اخیر با وجود قیمت پایین‌تر از میانگین ۳۰ روزه"
        else:
            technical_reason = "قرارگیری قیمت پایین‌تر از میانگین متحرک ۳۰ روزه و حجم معاملات نسبتا کم"

        trend_word = "صعودی" if direction_up else "نزولی"

        sentiment_effect = "تأیید" if (sentiment["score"] > 0 and direction_up) or (sentiment["score"] < 0 and not direction_up) else "تضعیف"

        text = (
            f"**قیمت پیش‌بینی‌شده برای روز بعد:** {predicted_price:,.2f}\n\n"
            f"**روند کلی:** {trend_word} (تغییر {change:+,.2f} معادل {change_pct:+.2f}٪)، "
            f"با سطح اطمینان تقریبی {confidence:.0f}٪.\n\n"
            f"**دلیل اصلی فنی روند:** به دلیل {technical_reason}.\n\n"
            f"**نقش تحلیل احساسات بازار:** تحلیل احساسات بازار با امتیاز {sentiment['score']:+.2f} "
            f"({sentiment['label']}) این روند را {sentiment_effect} می‌کند.\n\n"
            f"**هشدار:** این یک پیش‌بینی آماری بر اساس الگوهای گذشته است و سیگنال قطعی برای "
            f"تصمیم‌گیری سرمایه‌گذاری محسوب نمی‌شود. لطفاً پیش از هرگونه تصمیم مالی با یک "
            f"متخصص مشورت کنید."
        )
    else:
        if above_ma30 and high_volume:
            technical_reason = "the price sitting above its 30-day moving average combined with high trading volume"
        elif above_ma30:
            technical_reason = "the price sitting above its 30-day moving average"
        elif high_volume:
            technical_reason = "elevated recent trading volume despite the price being below its 30-day average"
        else:
            technical_reason = "the price sitting below its 30-day moving average with relatively low volume"

        trend_word = "upward" if direction_up else "downward"
        sentiment_effect = "confirms" if (sentiment["score"] > 0 and direction_up) or (sentiment["score"] < 0 and not direction_up) else "weakens"

        text = (
            f"**Predicted price for next day:** {predicted_price:,.2f}\n\n"
            f"**Overall trend:** {trend_word} ({change:+,.2f}, {change_pct:+.2f}%), "
            f"with an approximate confidence level of {confidence:.0f}%.\n\n"
            f"**Main technical driver:** {technical_reason}.\n\n"
            f"**Role of market sentiment:** Market sentiment analysis, scoring {sentiment['score']:+.2f} "
            f"({sentiment['label_en']}), {sentiment_effect} this trend.\n\n"
            f"**Disclaimer:** This is a statistical forecast based on historical patterns and is "
            f"not a definitive investment signal. Please consult a professional before making any "
            f"financial decisions."
        )

    return text


def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj



## تحلیل اکتشافی و آماری (EDA)



In [17]:

df_raw = load_raw_csv(PATHS["raw_csv"])
print("تعداد ردیف‌ها:", len(df_raw))
print("بازه‌ی زمانی:", df_raw["Date"].min().date(), "تا", df_raw["Date"].max().date())
df_raw.head()


تعداد ردیف‌ها: 2539
بازه‌ی زمانی: 2013-08-19 تا 2023-08-17


,Date,Close/Last,Volume,Open,High,Low
0,2013-08-19,1365.7,116056.0,1377.1,1384.1,1362.0
1,2013-08-20,1372.6,130096.0,1364.9,1378.0,1351.6
2,2013-08-21,1370.1,137350.0,1371.0,1378.9,1359.2
3,2013-08-22,1370.8,134493.0,1365.6,1381.4,1354.5
4,2013-08-23,1395.8,149116.0,1376.1,1399.9,1367.8


In [18]:

desc = df_raw[RAW_NUMERIC_COLS].describe().T
desc["missing_count"] = df_raw[RAW_NUMERIC_COLS].isna().sum()
desc.style.format("{:.2f}").set_caption("آمار توصیفی ستون‌های عددی")


,count,mean,std,min,25%,50%,75%,max,missing_count
Close/Last,2539.00,1467.44,282.89,1049.60,1243.90,1321.40,1774.05,2069.40,0.00
Volume,2511.00,183765.29,98028.94,1.00,123166.50,172127.00,233415.00,787217.00,28.00
Open,2539.00,1467.46,283.13,1051.50,1243.85,1321.70,1773.95,2076.40,0.00
High,2539.00,1477.04,285.23,1062.70,1251.25,1329.30,1785.00,2085.40,0.00
Low,2539.00,1457.63,280.37,1045.40,1235.80,1314.00,1763.55,2049.00,0.00


###  گزارش Missing Values و پاکسازی

In [19]:

report = missing_value_report(df_raw, RAW_NUMERIC_COLS)
print(report.to_string(index=False))


      ستون  تعداد گمشده  درصد گمشده
Close/Last            0       0.000
    Volume           28       1.103
      Open            0       0.000
      High            0       0.000
       Low            0       0.000


In [20]:

df_clean = clean_missing_values(df_raw)
print("مجموع NaN باقی‌مانده پس از پاکسازی:", int(df_clean[RAW_NUMERIC_COLS].isna().sum().sum()))


مجموع NaN باقی‌مانده پس از پاکسازی: 0


### Rolling Statistics (میانگین متحرک ۷ و ۳۰ روزه)

In [21]:

df_eda = add_rolling_stats_for_eda(df_clean)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_eda["Date"], y=df_eda[TARGET_COL], name="قیمت واقعی (Close/Last)", line=dict(width=1)))
fig.add_trace(go.Scatter(x=df_eda["Date"], y=df_eda["MA_7"], name="میانگین متحرک ۷ روزه", line=dict(width=1.5)))
fig.add_trace(go.Scatter(x=df_eda["Date"], y=df_eda["MA_30"], name="میانگین متحرک ۳۰ روزه", line=dict(width=2)))
fig.update_layout(title="قیمت طلا و میانگین‌های متحرک", xaxis_title="تاریخ", yaxis_title="قیمت", template="plotly_white", height=450)
fig.show()


###  تجمیع ۳۰ روزه 

In [22]:

df_ts = df_clean.set_index("Date")
resampled = df_ts.resample("30D").agg({TARGET_COL: "mean", "Volume": "sum"}).dropna()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=resampled.index, y=resampled[TARGET_COL], name="میانگین قیمت (۳۰ روزه)", line=dict(color="goldenrod", width=2)), secondary_y=False)
fig.add_trace(go.Bar(x=resampled.index, y=resampled["Volume"], name="مجموع حجم (۳۰ روزه)", opacity=0.4), secondary_y=True)
fig.update_layout(title="تجمیع ۳۰ روزه: میانگین قیمت در برابر مجموع حجم", template="plotly_white", height=450)
fig.update_yaxes(title_text="میانگین قیمت", secondary_y=False)
fig.update_yaxes(title_text="مجموع حجم", secondary_y=True)
fig.show()


### ماتریس همبستگی

In [23]:

corr = df_clean[RAW_NUMERIC_COLS].corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                 title="ماتریس همبستگی بین ستون‌های عددی")
fig.update_layout(height=500, template="plotly_white")
fig.show()


###  تحلیل روند  و تجزیه سری زمانی 

In [24]:

x_numeric = np.arange(len(df_clean)).reshape(-1, 1)
lr = LinearRegression().fit(x_numeric, df_clean[TARGET_COL].values)
trend_line = lr.predict(x_numeric)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_clean["Date"], y=df_clean[TARGET_COL], name="قیمت واقعی", line=dict(width=1)))
fig.add_trace(go.Scatter(x=df_clean["Date"], y=trend_line, name="خط روند (رگرسیون خطی)", line=dict(color="red", dash="dash", width=2)))
fig.update_layout(title="روند خطی قیمت طلا", template="plotly_white", height=450)
fig.show()

print(f"شیب روند: {lr.coef_[0]:.4f} واحد قیمت به ازای هر روز")


شیب روند: 0.3305 واحد قیمت به ازای هر روز


In [25]:

decomposition = seasonal_decompose(df_clean.set_index("Date")[TARGET_COL], model="additive", period=30, extrapolate_trend="freq")

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                     subplot_titles=("سری اصلی", "روند (Trend)", "فصلی (Seasonal)", "باقی‌مانده (Residual)"))
fig.add_trace(go.Scatter(x=decomposition.observed.index, y=decomposition.observed, name="Observed"), row=1, col=1)
fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend, name="Trend"), row=2, col=1)
fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal, name="Seasonal"), row=3, col=1)
fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid, name="Residual"), row=4, col=1)
fig.update_layout(height=800, title="تجزیه‌ی سری زمانی (دوره ۳۰ روزه)", template="plotly_white", showlegend=False)
fig.show()



##  مهندسی ویژگی و آماده‌ سازی برای مدل‌ سازی


In [26]:

features_df = build_feature_frame(df_clean)
print("شکل داده پس از فیچرینگ و حذف NaN:", features_df.shape)
features_df.to_csv(PATHS["features_csv"], index=False)
features_df.tail()


شکل داده پس از فیچرینگ و حذف NaN: (2509, 16)


,Date,Close/Last,Volume,Open,High,Low,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,daily_return,hl_ratio,day_of_week,month,day_of_year
2504,2023-08-11,1946.6,119090.0,1944.9,1953.6,1942.7,1948.9,1937.4,1917.9,1949.657143,12.533137,-0.118015,1.005611,4,8,223
2505,2023-08-14,1944.0,117514.0,1945.6,1948.2,1934.2,1946.6,1932.0,1929.4,1951.371429,10.345162,-0.133566,1.007238,0,8,226
2506,2023-08-15,1935.2,161512.0,1939.4,1944.3,1927.5,1944.0,1939.6,1929.5,1950.742857,11.271773,-0.452675,1.008716,1,8,227
2507,2023-08-16,1928.3,124766.0,1933.1,1938.2,1922.0,1935.2,1970.0,1927.1,1944.785714,10.382265,-0.356552,1.008429,2,8,228
2508,2023-08-17,1915.2,146770.0,1922.4,1933.5,1914.2,1928.3,1959.9,1915.4,1938.400000,12.962639,-0.679355,1.010083,3,8,229


In [27]:

train_df, test_df = train_test_split_timeseries(features_df, train_ratio=0.8)
print(f"Train: {train_df.shape}  ({train_df['Date'].min().date()} تا {train_df['Date'].max().date()})")
print(f"Test:  {test_df.shape}  ({test_df['Date'].min().date()} تا {test_df['Date'].max().date()})")

FEATURE_COLS = get_model_feature_columns()
with open(PATHS["feature_columns_path"], "w", encoding="utf-8") as f:
    json.dump(FEATURE_COLS, f, ensure_ascii=False, indent=2)
print("ستون‌های ویژگی:", FEATURE_COLS)


Train: (2007, 16)  (2013-09-30 تا 2021-08-18)
Test:  (502, 16)  (2021-08-19 تا 2023-08-17)
ستون‌های ویژگی: ['Open', 'High', 'Low', 'Volume', 'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'daily_return', 'hl_ratio', 'day_of_week', 'month', 'day_of_year']



### مقیاس‌ سازی (MinMaxScaler)



In [ ]:

NN_FEATURE_COLS = [TARGET_COL] + FEATURE_COLS  

if os.path.exists(PATHS["scaler_path"]):
    scaler = joblib.load(PATHS["scaler_path"])
    print("اسکیلر از قبل موجود بارگذاری شد (resume).")
else:
    scaler = MinMaxScaler()
    scaler.fit(train_df[NN_FEATURE_COLS].values)
    joblib.dump(scaler, PATHS["scaler_path"])
    print("اسکیلر جدید فیت و ذخیره شد.")

train_scaled = scaler.transform(train_df[NN_FEATURE_COLS].values)
test_scaled = scaler.transform(test_df[NN_FEATURE_COLS].values)

TARGET_IDX_IN_SCALED = 0  
SEQ_LEN = 30

X_train_full, y_train_full = create_sequences(train_scaled, seq_length=SEQ_LEN, target_idx=TARGET_IDX_IN_SCALED)

val_split_idx = int(len(X_train_full) * 0.9)
X_train, y_train = X_train_full[:val_split_idx], y_train_full[:val_split_idx]
X_val, y_val = X_train_full[val_split_idx:], y_train_full[val_split_idx:]

combined_scaled = np.vstack([train_scaled[-SEQ_LEN:], test_scaled])
X_test, y_test = create_sequences(combined_scaled, seq_length=SEQ_LEN, target_idx=TARGET_IDX_IN_SCALED)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)


اسکیلر جدید فیت و ذخیره شد.
X_train: (1779, 30, 15) X_val: (198, 30, 15) X_test: (502, 30, 15)



##  مدل LSTM

###  Optuna Hyperparameter Tuning (  مناسب برای اجرا بدون GPU)



In [ ]:

N_TRIALS_LSTM = 10      
MAX_EPOCHS_TUNING = 15   
BATCH_SIZE = 32

def build_lstm_model(units_1, units_2, dropout, learning_rate, n_features):
    model = Sequential([
        LSTM(units_1, return_sequences=True, input_shape=(SEQ_LEN, n_features)),
        Dropout(dropout),
        LSTM(units_2, return_sequences=False),
        Dropout(dropout),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mse")
    return model


def lstm_objective(trial):
    units_1 = trial.suggest_categorical("units_1", [32, 64, 96])
    units_2 = trial.suggest_categorical("units_2", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    model = build_lstm_model(units_1, units_2, dropout, learning_rate, X_train.shape[2])

    es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS_TUNING,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=0,
    )
    val_loss = min(history.history["val_loss"])
    return val_loss


lstm_storage = f"sqlite:///{PATHS['optuna_lstm_db']}"
lstm_study = optuna.create_study(
    study_name="lstm_gold_price",
    direction="minimize",
    storage=lstm_storage,
    sampler=TPESampler(seed=42),
    load_if_exists=True,
)

n_done = len(lstm_study.trials)
n_remaining = max(0, N_TRIALS_LSTM - n_done)
print(f"Trial های انجام‌شده قبلی: {n_done} | Trial های باقی‌مانده: {n_remaining}")

if n_remaining > 0:
    lstm_study.optimize(lstm_objective, n_trials=n_remaining, show_progress_bar=True)
else:
    print("تعداد trial کافی از قبل انجام شده؛ tuning رد شد (resume).")

print("بهترین پارامترهای LSTM:", lstm_study.best_params)
with open(PATHS["lstm_best_params_path"], "w") as f:
    json.dump(lstm_study.best_params, f, indent=2)


Trial های انجام‌شده قبلی: 0 | Trial های باقی‌مانده: 10


  0%|          | 0/10 [00:00<?, ?it/s]

بهترین پارامترهای LSTM: {'units_1': 96, 'units_2': 32, 'dropout': 0.2793699936433256, 'learning_rate': 0.0069782812651260325}


###  آموزش نهایی مدل LSTM با بهترین پارامترها

In [30]:

if os.path.exists(PATHS["lstm_model_path"]):
    print("مدل LSTM از قبل آموزش دیده — بارگذاری می‌شود (resume).")
    lstm_model = load_model(PATHS["lstm_model_path"])
    with open(PATHS["lstm_history_path"], "r") as f:
        lstm_history_dict = json.load(f)
else:
    best = lstm_study.best_params
    lstm_model = build_lstm_model(best["units_1"], best["units_2"], best["dropout"], best["learning_rate"], X_train.shape[2])

    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)

    progress_placeholder = st_progress_stub = None
    print("در حال آموزش نهایی LSTM ...")
    history = lstm_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=1,
    )
    lstm_model.save(PATHS["lstm_model_path"])
    lstm_history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(PATHS["lstm_history_path"], "w") as f:
        json.dump(lstm_history_dict, f)
    print("مدل LSTM ذخیره شد:", PATHS["lstm_model_path"])


در حال آموزش نهایی LSTM ...
Epoch 1/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0229 - val_loss: 0.0054 - learning_rate: 0.0070
Epoch 2/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0029 - val_loss: 0.0013 - learning_rate: 0.0070
Epoch 3/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0021 - val_loss: 0.0043 - learning_rate: 0.0070
Epoch 4/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0021 - val_loss: 0.0073 - learning_rate: 0.0070
Epoch 5/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0021 - val_loss: 0.0021 - learning_rate: 0.0070
Epoch 6/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0015 - val_loss: 0.0012 - learning_rate: 0.0035
Epoch 7/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0015 - val_loss: 9.4791e-04 - learning_rate: 0.0035
Epoch 8/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0014 - val_loss: 8.8179e-04 - learning_rate: 0.0035
Epoch 9/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0012 - val_loss: 9.0420e-04 - l

In [31]:

fig = go.Figure()
fig.add_trace(go.Scatter(y=lstm_history_dict["loss"], name="Train Loss"))
fig.add_trace(go.Scatter(y=lstm_history_dict["val_loss"], name="Validation Loss"))
fig.update_layout(title="تاریخچه Loss — مدل LSTM", xaxis_title="Epoch", yaxis_title="MSE Loss", template="plotly_white", height=400)
fig.show()



## مدل GRU

معماری مشابه LSTM، با لایه‌ های GRU به‌ جای LSTM.


In [32]:

N_TRIALS_GRU = 10

def build_gru_model(units_1, units_2, dropout, learning_rate, n_features):
    model = Sequential([
        GRU(units_1, return_sequences=True, input_shape=(SEQ_LEN, n_features)),
        Dropout(dropout),
        GRU(units_2, return_sequences=False),
        Dropout(dropout),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mse")
    return model


def gru_objective(trial):
    units_1 = trial.suggest_categorical("units_1", [32, 64, 96])
    units_2 = trial.suggest_categorical("units_2", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    model = build_gru_model(units_1, units_2, dropout, learning_rate, X_train.shape[2])

    es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS_TUNING,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=0,
    )
    return min(history.history["val_loss"])


gru_storage = f"sqlite:///{PATHS['optuna_gru_db']}"
gru_study = optuna.create_study(
    study_name="gru_gold_price",
    direction="minimize",
    storage=gru_storage,
    sampler=TPESampler(seed=42),
    load_if_exists=True,
)

n_done = len(gru_study.trials)
n_remaining = max(0, N_TRIALS_GRU - n_done)
print(f"Trial های انجام‌شده قبلی: {n_done} | Trial های باقی‌مانده: {n_remaining}")

if n_remaining > 0:
    gru_study.optimize(gru_objective, n_trials=n_remaining, show_progress_bar=True)
else:
    print("تعداد trial کافی از قبل انجام شده؛ tuning رد شد (resume).")

print("بهترین پارامترهای GRU:", gru_study.best_params)
with open(PATHS["gru_best_params_path"], "w") as f:
    json.dump(gru_study.best_params, f, indent=2)


Trial های انجام‌شده قبلی: 0 | Trial های باقی‌مانده: 10


  0%|          | 0/10 [00:00<?, ?it/s]

بهترین پارامترهای GRU: {'units_1': 64, 'units_2': 16, 'dropout': 0.11742508365045984, 'learning_rate': 0.005399484409787433}


In [33]:

if os.path.exists(PATHS["gru_model_path"]):
    print("مدل GRU از قبل آموزش دیده — بارگذاری می‌شود (resume).")
    gru_model = load_model(PATHS["gru_model_path"])
    with open(PATHS["gru_history_path"], "r") as f:
        gru_history_dict = json.load(f)
else:
    best = gru_study.best_params
    gru_model = build_gru_model(best["units_1"], best["units_2"], best["dropout"], best["learning_rate"], X_train.shape[2])

    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)

    print("در حال آموزش نهایی GRU ...")
    history = gru_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=1,
    )
    gru_model.save(PATHS["gru_model_path"])
    gru_history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(PATHS["gru_history_path"], "w") as f:
        json.dump(gru_history_dict, f)
    print("مدل GRU ذخیره شد:", PATHS["gru_model_path"])


در حال آموزش نهایی GRU ...
Epoch 1/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0083 - val_loss: 0.0020 - learning_rate: 0.0054
Epoch 2/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0020 - val_loss: 0.0012 - learning_rate: 0.0054
Epoch 3/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0016 - val_loss: 0.0015 - learning_rate: 0.0054
Epoch 4/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0013 - val_loss: 0.0040 - learning_rate: 0.0054
Epoch 5/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 9.7185e-04 - val_loss: 0.0017 - learning_rate: 0.0054
Epoch 6/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.9465e-04 - val_loss: 0.0046 - learning_rate: 0.0027
Epoch 7/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 9.4356e-04 - val_loss: 0.0028 - learning_rate: 0.0027
مدل GRU ذخیره شد: e:\term8\DM\prj3\gold_price_project\models\gru_model.keras


In [34]:

fig = go.Figure()
fig.add_trace(go.Scatter(y=gru_history_dict["loss"], name="Train Loss"))
fig.add_trace(go.Scatter(y=gru_history_dict["val_loss"], name="Validation Loss"))
fig.update_layout(title="تاریخچه Loss — مدل GRU", xaxis_title="Epoch", yaxis_title="MSE Loss", template="plotly_white", height=400)
fig.show()



##  مدل XGBoost


In [ ]:

X_train_xgb = train_df[FEATURE_COLS].values
y_train_xgb = train_df[TARGET_COL].values
X_test_xgb = test_df[FEATURE_COLS].values
y_test_xgb = test_df[TARGET_COL].values

tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}

if os.path.exists(PATHS["xgb_model_path"]) and os.path.exists(PATHS["xgb_best_params_path"]):
    print("مدل XGBoost از قبل آموزش دیده — بارگذاری می‌شود (resume).")
    xgb_model = xgb.XGBRegressor()
    xgb_model.load_model(PATHS["xgb_model_path"])
    with open(PATHS["xgb_best_params_path"], "r") as f:
        best_xgb_params = json.load(f)
else:
    base_model = xgb.XGBRegressor(objective="reg:squarederror", eval_metric="rmse", random_state=42)
    grid_search = GridSearchCV(base_model, param_grid, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    print("در حال اجرای GridSearchCV برای XGBoost (با TimeSeriesSplit, 3 splits) ...")
    grid_search.fit(X_train_xgb, y_train_xgb)

    best_xgb_params = grid_search.best_params_
    print("بهترین پارامترهای XGBoost:", best_xgb_params)

    xgb_model = xgb.XGBRegressor(objective="reg:squarederror", eval_metric="rmse", random_state=42, **best_xgb_params)
    xgb_model.fit(X_train_xgb, y_train_xgb)
    xgb_model.save_model(PATHS["xgb_model_path"])
    with open(PATHS["xgb_best_params_path"], "w") as f:
        json.dump(best_xgb_params, f, indent=2)
    print("مدل XGBoost ذخیره شد:", PATHS["xgb_model_path"])

cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train_xgb)):
    m = xgb.XGBRegressor(objective="reg:squarederror", eval_metric="rmse", random_state=42, **best_xgb_params)
    m.fit(X_train_xgb[tr_idx], y_train_xgb[tr_idx])
    preds = m.predict(X_train_xgb[val_idx])
    rmse_fold = compute_metrics(y_train_xgb[val_idx], preds)["RMSE"]
    cv_scores.append(rmse_fold)
    print(f"Fold {fold+1} RMSE: {rmse_fold:.3f}")

print(f"میانگین RMSE کراس-ولیدیشن: {np.mean(cv_scores):.3f}")


در حال اجرای GridSearchCV برای XGBoost (با TimeSeriesSplit, 3 splits) ...
بهترین پارامترهای XGBoost: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
مدل XGBoost ذخیره شد: e:\term8\DM\prj3\gold_price_project\models\xgb_model.json
Fold 1 RMSE: 8.021
Fold 2 RMSE: 26.879
Fold 3 RMSE: 273.762
میانگین RMSE کراس-ولیدیشن: 102.887



##  ارزیابی نهایی و مقایسه‌ ی سه مدل روی Test Set


In [36]:

# --- LSTM ---
lstm_pred_scaled = lstm_model.predict(X_test, verbose=0).ravel()

def inverse_target(scaled_target_values):
    dummy = np.zeros((len(scaled_target_values), len(NN_FEATURE_COLS)))
    dummy[:, TARGET_IDX_IN_SCALED] = scaled_target_values
    return scaler.inverse_transform(dummy)[:, TARGET_IDX_IN_SCALED]

lstm_pred = inverse_target(lstm_pred_scaled)
y_test_actual = inverse_target(y_test)

lstm_metrics = compute_metrics(y_test_actual, lstm_pred)
print("LSTM:", lstm_metrics)

# --- GRU ---
gru_pred_scaled = gru_model.predict(X_test, verbose=0).ravel()
gru_pred = inverse_target(gru_pred_scaled)
gru_metrics = compute_metrics(y_test_actual, gru_pred)
print("GRU:", gru_metrics)

# --- XGBoost ---
xgb_pred = xgb_model.predict(X_test_xgb)
xgb_metrics = compute_metrics(y_test_xgb, xgb_pred)
print("XGBoost:", xgb_metrics)


LSTM: {'MAE': 23.692100365893303, 'RMSE': 30.331189962564206, 'MAPE': 1.2716146899013883}
GRU: {'MAE': 29.74500697741945, 'RMSE': 37.308800625140925, 'MAPE': 1.5924726498045576}
XGBoost: {'MAE': 6.281158033880105, 'RMSE': 8.666726200137203, 'MAPE': 0.33699691686956057}


In [37]:

comparison = pd.DataFrame({
    "Model": ["LSTM", "GRU", "XGBoost"],
    "MAE": [lstm_metrics["MAE"], gru_metrics["MAE"], xgb_metrics["MAE"]],
    "RMSE": [lstm_metrics["RMSE"], gru_metrics["RMSE"], xgb_metrics["RMSE"]],
    "MAPE (%)": [lstm_metrics["MAPE"], gru_metrics["MAPE"], xgb_metrics["MAPE"]],
})
comparison = comparison.sort_values("RMSE").reset_index(drop=True)
best_model_name = comparison.iloc[0]["Model"]
print(f"بهترین مدل بر اساس RMSE: {best_model_name}")
comparison


بهترین مدل بر اساس RMSE: XGBoost


,Model,MAE,RMSE,MAPE (%)
0,XGBoost,6.281158,8.666726,0.336997
1,LSTM,23.692100,30.331190,1.271615
2,GRU,29.745007,37.308801,1.592473


In [38]:

# ذخیره‌ی معیارها و پیش‌بینی‌های Test برای استفاده در app.py
all_metrics = {
    "LSTM": lstm_metrics,
    "GRU": gru_metrics,
    "XGBoost": xgb_metrics,
    "best_model": best_model_name,
}
with open(PATHS["metrics_path"], "w", encoding="utf-8") as f:
    json.dump(to_json_safe(all_metrics), f, ensure_ascii=False, indent=2)

test_predictions = {
    "dates": test_df["Date"].dt.strftime("%Y-%m-%d").tolist(),
    "y_actual": y_test_actual.tolist() if best_model_name != "XGBoost" else y_test_xgb.tolist(),
    "LSTM": lstm_pred.tolist(),
    "GRU": gru_pred.tolist(),
    "XGBoost": xgb_pred.tolist(),
}
with open(PATHS["test_predictions_path"], "w", encoding="utf-8") as f:
    json.dump(to_json_safe(test_predictions), f, ensure_ascii=False)

train_meta = {
    "seq_len": SEQ_LEN,
    "nn_feature_cols": NN_FEATURE_COLS,
    "target_idx_in_scaled": TARGET_IDX_IN_SCALED,
    "feature_cols": FEATURE_COLS,
    "trained_at": pd.Timestamp.now().isoformat(),
}
with open(PATHS["train_meta_path"], "w", encoding="utf-8") as f:
    json.dump(train_meta, f, ensure_ascii=False, indent=2)

print("تمام مصنوعات (مدل‌ها، اسکیلر، متریک‌ها، پیش‌بینی‌های تست، متادیتا) ذخیره شدند.")
print("مسیر پروژه:", PROJECT_DIR)


تمام مصنوعات (مدل‌ها، اسکیلر، متریک‌ها، پیش‌بینی‌های تست، متادیتا) در Drive ذخیره شدند.
مسیر پروژه: e:\term8\DM\prj3\gold_price_project


In [39]:

fig = go.Figure()
best_pred = {"LSTM": lstm_pred, "GRU": gru_pred, "XGBoost": xgb_pred}[best_model_name]
best_actual = y_test_actual if best_model_name != "XGBoost" else y_test_xgb

fig.add_trace(go.Scatter(y=best_actual, name="واقعی", line=dict(width=2)))
fig.add_trace(go.Scatter(y=best_pred, name=f"پیش‌بینی ({best_model_name})", line=dict(width=2, dash="dot")))
fig.update_layout(title=f"پیش‌بینی در برابر واقعیت — بهترین مدل: {best_model_name}", template="plotly_white", height=450)
fig.show()



## پایان آموزش

همه‌ی مصنوعات لازم برای اپ Streamlit در پوشه‌ی محلی `gold_price_project/`
(کنار این نوت‌بوک) ذخیره شدند:
- مدل‌های `lstm_model.keras`, `gru_model.keras`, `xgb_model.json`
- اسکیلر `minmax_scaler.pkl`
- متریک‌ها و مقایسه‌ی مدل‌ها `metrics.json`
- پیش‌بینی‌های Test برای رسم نمودار `test_predictions.json`
- متادیتای آموزش `train_meta.json`

اکنون می‌توانید `app.py` را با `streamlit run app.py` از همین پوشه اجرا کنید.
